# 長江杯 2026 — Target 7 (`IL-7Rα` / `CD127`) 雙開物理隔離專版 A100 Colab Worker (`target7-il7ra` 分支)

**與正在運行的 Target 5 (`main` 分支) 100% 物理隔離、零互擾設計**：
- 🔒 **獨立分支與本地目錄**：鎖定拉取 `target7-il7ra` 分支至 `/content/changjiang-protein-2026-t7`。
- 📂 **獨立網盤隊列與註冊表**：僅讀寫共享網盤下的 `jobs_target7_IL7Ra/{pending,running,completed,failed}/` 與 `registry/design_registry_T7_IL7Ra.csv`，**絕不觸碰 T5 正在使用的 `jobs/` 與 `design_registry.csv`**！
- ⚡ **安全唯讀共享 45 秒環境快取**：直接唯讀解壓網盤已有的 `cache/bindcraft_colab_cache.tar.zst` 至本地 NVMe SSD（45 秒開跑），並將 HuggingFace ESMFold 權重複製至本地 `/content/hf_cache`，消除雙 Colab 網盤 `.locks/` 鎖競爭。
- 🧬 **內建 Target 7 專屬硬閘**：`SOP §B-1 --strict-sidechains` 預檢、`SolubleMPNN` (`omit_AAs=CM`)、`40–100 aa` 長度硬限、與 `7OPB_D`/`IL-7`/`TSLP` 同一性 `<= 70%` 新穎性硬閘、`3DI3` 晶體三條糖鏈 (`Chains C/D/E`) `< 2.5 Å` 碰撞攔截 + `IL-7` 正交阻斷驗證、以及對完整官方 `219 aa (Glu21–Asp239)` 靶點的 `ESMFold + ipSAE` 與 **`1:1` AlphaFold 3 Server JSON** 自動導出！

In [ ]:
#@title 1. 檢查 GPU 架構、掛載共享 Google Drive、同步 `target7-il7ra` 分支並印出 T7 獨立隊列診斷
import os, sys, pathlib, importlib, importlib.util, torch
from google.colab import drive

SHARED_DRIVE_URL = "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
GIT_BRANCH = "target7-il7ra" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026-t7"

print("=== [1/4] GPU 架構與顯存自檢 ===")
!nvidia-smi
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1e9
    sm = f"sm_{p.major}{p.minor}"
    print(f"檢測到 GPU: {p.name} | 顯存: {gb:.1f} GB | 架構: {sm}")
    if p.major >= 12 or "BLACKWELL" in p.name.upper() or "RTX PRO 6000" in p.name.upper():
        print("❌ [警告] 當前為 Blackwell 架構，BindCraft (CUDA 12.4/JAX) 不支持，請切換為 A100！")
    else:
        print("✅ GPU 架構兼容 BindCraft + SolubleMPNN + ESMFold（IL-7Rα 核心域僅 193 aa，速度比三聚體快約 2 倍！）✓")
else:
    print("❌ 未檢測到 GPU！請在『執行階段 -> 變更執行階段類型』選擇 A100 GPU。")

print("\n=== [2/4] 安裝 ESMFold / ipSAE 評估依賴 ===")
!pip install -q transformers accelerate

print("\n=== [3/4] 掛載 Google Drive ===")
if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✅ Google Drive 已掛載！")

print(f"\n=== [4/4] 同步 GitHub 分支 `{GIT_BRANCH}` 並連接獨立 T7 隊列 ===")
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone -b {GIT_BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {GIT_BRANCH} && git -C {REPO_DIR} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}

%cd {REPO_DIR}
sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker_target7", f"{REPO_DIR}/pipeline/worker_target7.py"),
]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
wt7 = sys.modules["worker_target7"]

DRIVE_WORKSPACE = str(wt7.resolve_shared_drive_workspace(SHARED_DRIVE_URL))
print(f"🔗 已鎖定團隊共享網盤：{DRIVE_WORKSPACE}")
!python3 scripts/tnf_pipeline.py selftest
wt7.main(["--drive-root", DRIVE_WORKSPACE, "--diagnose"])

In [ ]:
#@title 2. 極速唯讀還原 BindCraft + SolubleMPNN + AF2 權重（共享 `bindcraft_colab_cache.tar.zst` 45 秒秒開，不覆寫 T5 快取！）
import os, sys, time, pathlib, importlib, importlib.util, torch
from google.colab import drive

SAVE_ENV_CACHE_IF_MISSING = True #@param {type:"boolean"}
SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
GIT_BRANCH = globals().get("GIT_BRANCH", "target7-il7ra")
REPO_DIR = "/content/changjiang-protein-2026-t7"

if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 12:
    raise RuntimeError("當前 GPU 為 Blackwell (sm_120)，與 BindCraft CUDA 12.4 不兼容！請切換為 A100 再執行。")

if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone -b {GIT_BRANCH} https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {GIT_BRANCH} && git -C {REPO_DIR} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}

for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker_target7", f"{REPO_DIR}/pipeline/worker_target7.py"),
]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
wt7 = sys.modules["worker_target7"]

shared_root = wt7.resolve_shared_drive_workspace(SHARED_DRIVE_URL)
cache_dir = shared_root / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_tar = cache_dir / "bindcraft_colab_cache.tar.zst"

bc_python = "/content/miniforge3/envs/BindCraft/bin/python"
bc_params_file = "/content/BindCraft/params/params_model_1_ptm.npz"

if (not os.path.exists(bc_python) or not os.path.exists(bc_params_file)) and cache_tar.exists():
    t0 = time.time()
    print(f"🚀 偵測到共享網盤已存在預編譯環境快取 ({cache_tar.name})！正在唯讀極速解壓至當前 Colab 本地 SSD（約 45–60 秒）...")
    !apt-get install -y -qq zstd
    !tar -I 'zstd -d -T0' -xf "{cache_tar}" -C /
    print(f"⚡ 環境快取還原完成，耗時 {time.time() - t0:.1f} 秒！")

if not os.path.exists("/content/miniforge3/bin/conda"):
    !wget -q -O /content/Miniforge3.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /content/Miniforge3.sh -b -p /content/miniforge3

if not os.path.exists("/content/BindCraft/bindcraft.py"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft

if not os.path.exists(bc_python) or not os.path.exists(bc_params_file):
    print("正在首次安裝 BindCraft 環境與下載 AlphaFold2 權重（約 8–12 分鐘）...")
    !cd /content/BindCraft && PATH="/content/miniforge3/bin:$PATH" bash install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'

if os.path.exists(bc_python) and os.path.exists(bc_params_file):
    !chmod +x /content/BindCraft/functions/dssp /content/BindCraft/functions/DAlphaBall.gcc
    print("✅ BindCraft + SolubleMPNN + PyRosetta + AlphaFold2 權重已完整就緒 ✓")
    if SAVE_ENV_CACHE_IF_MISSING and not cache_tar.exists():
        !apt-get install -y -qq zstd
        !tar -I 'zstd -1 -T0' -cf "/content/bindcraft_colab_cache_t7.tar.zst.tmp" /content/miniforge3 /content/BindCraft
        if not cache_tar.exists():
            !mv -n "/content/bindcraft_colab_cache_t7.tar.zst.tmp" "{cache_tar}"
        !rm -f "/content/bindcraft_colab_cache_t7.tar.zst.tmp"
else:
    raise RuntimeError("❌ BindCraft 環境未就緒，請檢查上方日誌。")

In [ ]:
#@title 3. 投遞 Target 7 (`IL-7Rα`) 任務單至獨立隊列 `jobs_target7_IL7Ra/pending/`（零影響 T5！）
import os, re, sys, json, pathlib, importlib.util
from google.colab import drive

# 推薦執行順序（SOP §D-2 多模板配額）：
#   1) target7_IL7Ra_core_3DI3_HS1 (預設主力 40%：3DI3 結合態核心域 + 無糖鏈位阻 HS1)
#   2) target7_IL7Ra_holo_7OPB_HS3_mixed (跨晶體交叉 25%：7OPB 實測構象 + random_helicity 混合 α/β)
#   3) target7_IL7Ra_apo_3UP1_HS4 (靜息態 20%：3UP1 無配體構象，確保識別天然靜息受體)
#   4) target7_IL7Ra_backMaskK_HS1 (背側封鎖 15%：N45K/N131K/N162K 負向設計緊湊型 50–75 aa)
TEMPLATE_CHOICE = "target7_IL7Ra_core_3DI3_HS1" #@param ["target7_IL7Ra_core_3DI3_HS1", "target7_IL7Ra_holo_7OPB_HS3_mixed", "target7_IL7Ra_apo_3UP1_HS4", "target7_IL7Ra_backMaskK_HS1", "none"]
MAX_TRAJECTORIES = 0 #@param {type:"integer"}
SUBMITTED_BY = "David" #@param {type:"string"}
SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
GIT_BRANCH = globals().get("GIT_BRANCH", "target7-il7ra")
REPO_DIR = "/content/changjiang-protein-2026-t7"

if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone -b {GIT_BRANCH} https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {GIT_BRANCH} && git -C {REPO_DIR} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}

for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker_target7", f"{REPO_DIR}/pipeline/worker_target7.py"),
]:
    if _mname not in sys.modules:
        _spec = importlib.util.spec_from_file_location(_mname, _mpath)
        _mod = importlib.util.module_from_spec(_spec)
        sys.modules[_mname] = _mod
        _spec.loader.exec_module(_mod)
wt7 = sys.modules["worker_target7"]

dirs = wt7.ensure_t7_workspace(wt7.resolve_shared_drive_workspace(SHARED_DRIVE_URL))
pending_dir = dirs["pending"]
running_dir = dirs["running"]
runs_dir = dirs["runs"]

mapping = {
    "target7_IL7Ra_core_3DI3_HS1": f"{REPO_DIR}/jobs/templates/job_target7_IL7Ra_minibinder.json",
    "target7_IL7Ra_holo_7OPB_HS3_mixed": f"{REPO_DIR}/jobs/templates/job_target7_IL7Ra_holo7OPB.json",
    "target7_IL7Ra_apo_3UP1_HS4": f"{REPO_DIR}/jobs/templates/job_target7_IL7Ra_apo3UP1.json",
    "target7_IL7Ra_backMaskK_HS1": f"{REPO_DIR}/jobs/templates/job_target7_IL7Ra_backMaskK.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    if (running_dir / src.name).exists():
        print(f"🔄 任務 {src.name} 已在獨立隊列 jobs_target7_IL7Ra/running/ 中，直接執行『第 4 格』即可續跑！")
    else:
        jdata = json.loads(src.read_text(encoding="utf-8"))
        if SUBMITTED_BY.strip():
            jdata["submitted_by"] = SUBMITTED_BY.strip()
        if int(MAX_TRAJECTORIES) > 0:
            jdata["max_trajectories"] = int(MAX_TRAJECTORIES)
        jid = str(jdata.get("job_id", src.stem))
        m = re.match(r"^(.*_)(\d{3,})$", jid)
        if m:
            prefix, num_s = m.group(1), m.group(2)
            num = int(num_s)
            while True:
                cand_id = f"{prefix}{num:0{len(num_s)}d}"
                mf = runs_dir / cand_id / "run_manifest.json"
                if mf.exists():
                    try:
                        if json.loads(mf.read_text(encoding="utf-8")).get("status") == "COMPLETED":
                            num += 1
                            continue
                    except Exception:
                        pass
                jid = cand_id
                break
            jdata["job_id"] = jid
        dst = pending_dir / src.name
        dst.write_text(json.dumps(jdata, indent=2, ensure_ascii=False), encoding="utf-8")
        (dirs["failed"] / src.name).unlink(missing_ok=True)
        print(f"✅ 已投遞 Target 7 任務單 (job_id={jdata['job_id']}, max_trajectories={jdata['max_trajectories']}) 至獨立隊列：{dst}")

In [ ]:
#@title 4. 啟動 Target 7 獨立隊列執行器（寫入 `design_registry_T7_IL7Ra.csv` + 自動導出 1:1 AF3 JSON）
import os, sys, time, importlib, importlib.util
from google.colab import runtime

AUTO_UNASSIGN_GPU = True #@param {type:"boolean"}
GIT_BRANCH = globals().get("GIT_BRANCH", "target7-il7ra")
REPO_DIR = "/content/changjiang-protein-2026-t7"

if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone -b {GIT_BRANCH} https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {GIT_BRANCH} && git -C {REPO_DIR} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}

sys.path_importer_cache.clear()
importlib.invalidate_caches()
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker_target7", f"{REPO_DIR}/pipeline/worker_target7.py"),
]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
wt7 = sys.modules["worker_target7"]

SHARED_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
DRIVE_WORKSPACE = str(wt7.resolve_shared_drive_workspace(SHARED_URL))
argv = ["--drive-root", DRIVE_WORKSPACE, "--once"]
if AUTO_UNASSIGN_GPU:
    argv.append("--unassign-on-exit")

exit_code = wt7.main(argv)

if AUTO_UNASSIGN_GPU and exit_code == 0:
    if hasattr(os, "sync"):
        os.sync()
    try:
        from google.colab import drive as _colab_drive
        _colab_drive.flush_and_unmount()
    except Exception:
        pass
    print("🔌 正在釋放當前 T7 Colab 的 GPU (不影響另一個 T5 Colab)...", flush=True)
    time.sleep(2)
    runtime.unassign()

In [ ]:
#@title 5. ⚡ 1 分鐘獨立採收增強與一鍵下載 `1:1 IL-7Rα` AlphaFold 3 Server JSON（T4/L4/A100 均可獨立秒跑）
import os, sys, time, pathlib, importlib, importlib.util
from google.colab import drive, files, runtime

JOB_ID = "job_t7_il7ra_core_001" #@param {type:"string"}
RUN_EXTRA_SOLUBLE_MPNN_HARVEST = True #@param {type:"boolean"}
MIN_TRAJ_IPTM = 0.72 #@param {type:"number"}
NUM_SEQS_CORE_FIX = 6 #@param {type:"integer"}
NUM_SEQS_FULL_REDESIGN = 6 #@param {type:"integer"}
OMIT_AAS = "CXM" #@param {type:"string"}
MAX_KEEP_PER_BACKBONE = 2 #@param {type:"integer"}
AUTO_DOWNLOAD_AF3_JSON = True #@param {type:"boolean"}
AUTO_UNASSIGN_GPU_AFTER = False #@param {type:"boolean"}

SHARED_DRIVE_URL = globals().get("SHARED_DRIVE_URL", "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link")
GIT_BRANCH = globals().get("GIT_BRANCH", "target7-il7ra")
REPO_DIR = "/content/changjiang-protein-2026-t7"

if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
if not os.path.exists(f"{REPO_DIR}/.git"):
    !rm -rf {REPO_DIR} && git clone -b {GIT_BRANCH} https://github.com/David-Liv/changjiang-protein-2026.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {GIT_BRANCH} && git -C {REPO_DIR} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}

for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)
for _mname, _mpath in [
    ("check_novelty", f"{REPO_DIR}/scripts/check_novelty.py"),
    ("tnf_pipeline", f"{REPO_DIR}/scripts/tnf_pipeline.py"),
    ("worker_target7", f"{REPO_DIR}/pipeline/worker_target7.py"),
]:
    _spec = importlib.util.spec_from_file_location(_mname, _mpath)
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules[_mname] = _mod
    _spec.loader.exec_module(_mod)
wt7 = sys.modules["worker_target7"]
drive_root = wt7.resolve_shared_drive_workspace(SHARED_DRIVE_URL)

if RUN_EXTRA_SOLUBLE_MPNN_HARVEST:
    wt7.main([
        "--drive-root", str(drive_root),
        "--harvest-job-id", JOB_ID.strip(),
        "--min-iptm", str(MIN_TRAJ_IPTM),
        "--num-seqs-core-fix", str(NUM_SEQS_CORE_FIX),
        "--num-seqs-full", str(NUM_SEQS_FULL_REDESIGN),
        "--omit-aas", OMIT_AAS.strip() or "CXM",
        "--max-per-backbone", str(MAX_KEEP_PER_BACKBONE),
    ])

af3_dir = drive_root / "runs" / JOB_ID.strip() / "af3_inputs"
af3_jsons = sorted(af3_dir.glob(f"af3_server_{JOB_ID.strip()}*.json")) if af3_dir.exists() else []
if AUTO_DOWNLOAD_AF3_JSON and af3_jsons:
    print(f"\n📥 正在觸發瀏覽器下載 {len(af3_jsons)} 個 1:1 (Minibinder + 219aa IL-7Rα) AlphaFold 3 JSON：")
    for jp in af3_jsons:
        print(f"   ⬇️ {jp.name}")
        files.download(str(jp))
        time.sleep(1.5)

if AUTO_UNASSIGN_GPU_AFTER:
    if hasattr(os, "sync"):
        os.sync()
    time.sleep(3)
    runtime.unassign()